# 03. Propositional Logic

Notebook ini membangun **propositional logic** dari nol: proposition symbol,
kelima logical connective, semantics lewat model, dan truth table. Semuanya
memakai kelas `Expr` dari `logic.py`, jembatan dari notasi model berbasis
`dict` di Notebook 02 menuju sentence yang bisa langsung dievaluasi lewat
kode. Notebook ini jadi dasar sebelum masuk forward/backward chaining dan
first-order logic di notebook-notebook berikutnya.

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. membedakan proposition symbol yang atomik dari complex sentence, dan
   menjelaskan kenapa symbol seperti `W12` tidak sama dengan pemanggilan
   predicate `W(1, 2)`;
2. memakai kelima logical connective ($\neg, \land, \lor, \Rightarrow,
   \Leftrightarrow$) lewat `Expr` dan `expr()`;
3. mengevaluasi nilai kebenaran sebuah sentence terhadap suatu model dengan
   `pl_true`; dan
4. membaca truth table untuk mengenali tautology, satisfiable, dan
   unsatisfiable, serta menjelaskan kenapa implikasi material tetap benar
   ketika premise-nya salah.


## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : C:\Users\cathl\Kuliah\KCV\KK\Modul-Praktikum-KK-RKA-25\logics\praktikum\environment
Python      : 3.14.2
Check       : tt_entails(P & Q, Q) = True


---
# 3.1 Syntax Logika Proposisional

## Penjelasan

**Propositional logic** (logika proposisi) adalah bentuk logika paling
sederhana untuk merepresentasikan pengetahuan (knowledge representation) dan
melakukan penalaran (reasoning). Kesederhanaan ini yang membuatnya pas jadi
titik awal: cukup untuk memperkenalkan cara kerja syntax, semantics, dan
inference, sebelum first-order logic yang lebih rumit dibahas setelah modul
ini.

**Syntax** adalah aturan yang menentukan sentence mana yang sah ditulis dalam
bahasa logika ini, bukan soal benar-salah maknanya, tapi soal sah-tidaknya
bentuknya, mirip aturan tata bahasa pemrograman yang menentukan kode mana
yang valid terlepas dari apa yang dilakukan kode itu. Syntax logika
proposisi mengenal dua bentuk sentence berikut.

- **Proposition symbol**\
Unit paling dasar. Satu simbol mewakili satu proposisi yang nilainya bisa
benar atau salah. Contoh dari slide: `P`, `Q`, `R`, `W12`, `North`. `P` dan
`W` di sini ikut konvensi penamaan Wumpus World yang dipakai sepanjang
modul: `P` untuk pit (lubang), `W` untuk wumpus, jadi `W12` dimaksudkan
sebagai "ada wumpus di kotak [1,2]". Ada juga dua simbol khusus yang
nilainya sudah tetap, `True` dan `False`.

**Mengecek apakah suatu kalimat pantas jadi proposition symbol**\
Cara praktisnya: tempelkan di depannya frasa "benar bahwa ...". Kalau
hasilnya tetap masuk akal secara gramatikal, kalimat itu proposisi. Misalnya,
"benar bahwa reaktor sedang menyala" masuk akal, jadi "reaktor sedang
menyala" adalah proposisi. Sebaliknya "benar bahwa apakah kamu akan pergi?"
tidak masuk akal (pertanyaan bukan proposisi karena tidak punya nilai
benar/salah). 

**Proposition symbol bersifat atomik**\
Proposition symbol tidak punya struktur di
dalamnya. `W12` cuma sebuah nama, bukan pemanggilan fungsi `W` dengan
argumen 1 dan 2. Walau namanya sengaja dibuat mirip koordinat kotak $[1,2]$
di Wumpus World supaya gampang diingat, propositional logic sendiri tidak
tahu itu. `W12` dan `W13` baginya cuma dua nama yang sama sekali tidak
berhubungan. First-order logic mengatasi keterbatasan ini lewat predicate
yang argumennya sungguhan, sehingga `W(1, 2)` dan `W(1, 3)` bisa dikenali
sebagai predicate yang sama dengan argumen berbeda.

- **Complex sentence**\
Dibangun dari sentence yang lebih sederhana memakai tanda kurung dan
konektif logika (logical connectives). Contoh singkat: $P \land Q$, atau
$\neg W12 \Rightarrow R$. Kelima konektifnya dibahas satu per satu di
sub-topik berikutnya.

**Sentence yang sah dan tidak sah**\
Syntax juga menentukan susunan kurung dan konektif mana yang sah. Kalau $P$,
$Q$, $R$, $S$ proposition symbol, maka $P \land Q \Rightarrow R$,
$P \land (Q \Rightarrow R)$, dan $(P \land (Q \Rightarrow R)) \lor S$
semuanya sentence yang sah, boleh bersarang sedalam apa pun asal kurungnya
seimbang. Sebaliknya `P ∧` (menggantung, kurang satu operand), `P ∧ Q)`
(kurung tidak seimbang), dan `P¬` (negasi di posisi salah) bukan sentence
yang sah.

## Contoh penerapan

Bagian ini memakai kelas `Expr` dari `logic.py`, dulu sebelum masuk ke
proposisi Wumpus. Simbol generiknya sengaja dipilih `x`, `y`, `P`, `Q`, `f`,
sama seperti contoh pada `logic.py`, supaya perhatian tertuju ke bentuknya dulu,
bukan ke makna logisnya.

In [2]:
# Symbol('x') builds a nullary Expr: op holds the name, args is empty.
# A lone proposition symbol has no internal structure: that empty
# args tuple is exactly what "atomic" means.
x = Symbol('x')

print(x)
print("op   :", repr(x.op))
print("args :", x.args)

x
op   : 'x'
args : ()


In [3]:
# expr() is not just for building sentences: a comma-separated string
# evaluates to a tuple, and every identifier inside is auto-defined as a
# Symbol. That makes it double as a shorthand for declaring several symbols
# at once, so every symbol in this notebook goes through the same expr()
# entry point instead of a separate symbols() helper.
x, y, P, Q, f = expr('x, y, P, Q, f')
print(x, y, P, Q, f)

x y P Q f


In [4]:
# Calling a Symbol builds a compound Expr: f(x, y) is really
# Symbol.__call__, giving Expr('f', x, y). Expr nests the same way, so
# ordinary Python arithmetic builds an expression tree for free.
nested = 3 * f(x, y) + P(y) / 2 + 1

print(nested)
print("op   :", nested.op)
print("args :", nested.args)

(((3 * f(x, y)) + (P(y) / 2)) + 1)
op   : +
args : (((3 * f(x, y)) + (P(y) / 2)), 1)


In [5]:
# Back to the slide's own example: W12 is one atomic name, not a call
# to W with arguments 1 and 2. Compare its empty args to nested.args
# above, which is exactly the atomic-vs-compound distinction from the
# Penjelasan.
W12 = Symbol('W12')
print(W12, "-> op:", repr(W12.op), " args:", W12.args)

W12 -> op: 'W12'  args: ()


---
# 3.2 Logical Connectives

![Penjelasan lima connective](img/slide-26-penjelasan-connectives.png)

## Penjelasan

**Logical connectives** adalah operator yang menggabungkan sentence menjadi
complex sentence. 

- **Negasi ($\neg$, NOT)**\
Membalik nilai kebenaran sentence di belakangnya. Contoh: $\neg W_{1,3}$
berarti "tidak ada wumpus di [1,3]", kebalikan dari $W_{1,3}$. Sentence
atomik tanpa negasi disebut *positive literal* (misalnya $W_{1,3}$), yang
dinegasi disebut *negative literal* (misalnya $\neg W_{1,3}$). Keduanya
sama-sama disebut literal.

- **Konjungsi ($\land$, AND)**\
Menggabungkan dua sentence, bernilai benar hanya kalau keduanya benar.
Bagian-bagian yang digabungkan disebut *conjunct*. Contoh:
$W_{1,3} \land P_{3,1}$ ("ada wumpus di [1,3] dan ada pit di [3,1]"), dengan
$W_{1,3}$ dan $P_{3,1}$ sebagai dua conjunct-nya.

- **Disjungsi ($\lor$, OR)**\
Menggabungkan dua sentence, bernilai benar kalau salah satu (atau keduanya)
benar. Bagian-bagiannya disebut *disjunct*. Contoh: $(W_{1,3} \land P_{3,1}) \lor W_{2,2}$.

- **Implikasi ($\Rightarrow$, IF-THEN)**\
Bagian kiri disebut *premise* atau *antecedent*, bagian kanan disebut
*conclusion* atau *consequent*. Implikasi juga dikenal sebagai *rule* atau
pernyataan if-then, dan buku lain kadang menulisnya sebagai $\supset$ atau
$\rightarrow$. Contoh: $(W_{1,3} \land P_{3,1}) \Rightarrow \neg W_{2,2}$
("kalau ada wumpus di [1,3] dan pit di [3,1], maka tidak ada wumpus di
[2,2]"). Implikasi ini yang jadi bentuk rule di KB nanti di Notebook 04, jadi
paling penting di antara kelima konektif. Detail perilakunya, termasuk dua
pertanyaan jebakan yang sering bikin bingung, dibahas di 3.4.

- **Bikondisional ($\Leftrightarrow$, IFF)**\
Dua arah sekaligus: $\alpha \Leftrightarrow \beta$ sama dengan
$(\alpha \Rightarrow \beta) \land (\beta \Rightarrow \alpha)$. Buku lain
menulisnya sebagai $\equiv$. Contoh: $W_{1,3} \Leftrightarrow \neg W_{2,2}$.

Beberapa sumber, termasuk source code, memakai notasi ASCII biasa sebagai
pengganti simbol matematis: `&` atau `.` untuk AND, `|` atau `+` untuk OR,
`~` untuk NOT. Bukan kebetulan operator Python yang dipakai `logic.py` nanti
di Contoh penerapan (`&`, `|`, `~`) persis mengikuti konvensi ASCII ini.

## Contoh penerapan

Tabel pemetaan antara notasi buku dan input Python:

| Operasi | Notasi Buku | Input Python | Output Python |
|---|---|---|---|
| Negasi | $\neg P$ | `~P` | `~P` |
| Konjungsi | $P \land Q$ | `P & Q` | `(P & Q)` |
| Disjungsi | $P \lor Q$ | `P \| Q` | `(P \| Q)` |
| Implikasi | $P \Rightarrow Q$ | `P \|'==>'\| Q` | `(P ==> Q)` |
| Bikondisional | $P \Leftrightarrow Q$ | `P \|'<=>'\| Q` | `(P <=> Q)` |

Python tidak mengizinkan `==>` sebagai operator, jadi `logic.py` menyediakan
trik `|'==>'|` sebagai penggantinya. Fungsi `expr()` membungkus trik itu:
dia menerima string biasa, otomatis mendefinisikan simbol yang muncul di
dalamnya, jadi triknya tidak perlu ditulis manual.

In [6]:
# Negation, conjunction, and disjunction have real Python operators.
P, Q = expr('P, Q')

print(~P)
print(P & Q)
print(P | Q)

# ==> and <=> are not valid Python operators. logic.py works around this
# with the |'==>'| trick: P |'==>'| Q first builds PartialExpr('==>', P),
# then combining that with Q via | finishes the Expr.
print(P |'==>'| Q)
print(P |'<=>'| Q)

~P
(P & Q)
(P | Q)
(P ==> Q)
(P <=> Q)


In [7]:
# expr() parses a string directly: ==>, <==, and <=> inside the string are
# rewritten to the |'...'| trick automatically, so no need to type it by
# hand. Any identifier that appears is auto-defined as a Symbol, even ones
# that were never assigned before: R and S below did not exist yet.
print(expr('P ==> Q'))
print(expr('P <=> Q'))
print(expr('R & S'))

(P ==> Q)
(P <=> Q)
(R & S)


In [8]:
# Precedence trap: ==> is rewritten to |'==>'|, which shares Python's |
# precedence. Without parentheses, the string does not parse the way it
# reads: the trailing "| Q" ends up attached to the wrong side.
wrong = expr('P & Q ==> P | Q')
right = expr('(P & Q) ==> (P | Q)')

print("wrong:", wrong)
print("right:", right)
print("same? ", wrong == right)

wrong: (((P & Q) ==> P) | Q)
right: ((P & Q) ==> (P | Q))
same?  False


In [9]:
# Same mapping as the table above, generated instead of typed by hand.
rows = [
    ("Negasi", "~P", repr(~P)),
    ("Konjungsi", "P & Q", repr(P & Q)),
    ("Disjungsi", "P | Q", repr(P | Q)),
    ("Implikasi", "P |'==>'| Q", repr(P |'==>'| Q)),
    ("Bikondisional", "P |'<=>'| Q", repr(P |'<=>'| Q)),
]
pd.DataFrame(rows, columns=["Operasi", "Input Python", "Output Python"])

,Operasi,Input Python,Output Python
0,Negasi,~P,~P
1,Konjungsi,P & Q,(P & Q)
2,Disjungsi,P | Q,(P | Q)
3,Implikasi,P |'==>'| Q,(P ==> Q)
4,Bikondisional,P |'<=>'| Q,(P <=> Q)


---
# 3.3 Semantics

## Penjelasan

**Semantics** adalah aturan untuk menentukan nilai kebenaran sebuah sentence,
benar atau salah, terhadap suatu model tertentu. Kalau syntax mengurus bentuk
mana yang sah ditulis, semantics mengurus arti dan nilainya.

**Model** ($m$) menetapkan nilai benar atau salah untuk **setiap** proposition
symbol yang dipakai KB: bukan sebagian, harus semua. Kalau KB memakai tiga
simbol $P_{1,2}$, $P_{2,2}$, $P_{3,1}$, salah satu model yang mungkin adalah:

$$m_1 = \{P_{1,2} = false,\ P_{2,2} = false,\ P_{3,1} = true\}$$

Ini bentuk yang sama dengan model di Notebook 02 (dict yang memetakan simbol
ke nilai), cuma sekarang key-nya berupa objek `Expr`, bukan string biasa.
Istilah ini juga sering disebut *valuation* di sumber lain, namanya beda,
konsepnya sama: fungsi yang memetakan tiap proposition symbol ke nilai
benar/salah.

Aturan tepat untuk menentukan nilai tiap konektif terhadap suatu model direkap
dalam truth table. Truth table dibahas satu per satu di 3.4; di sini cukup
pahami dulu bahwa model adalah "input"-nya dan nilai benar/salah adalah
"output"-nya.

## Contoh penerapan

Perkenalkan `pl_true(sentence, model)`, lihat source code-nya lewat
`psource()`. Di situ kelihatan dia cuma evaluasi rekursif biasa berdasarkan
`op` dan `args` sentence-nya.

Sebelum dicek dengan kode, hitung manual dulu contoh dari slide 29. Dengan
$m_1 = \{P_{1,2} = false, P_{2,2} = false, P_{3,1} = true\}$, berapa nilai
$\neg P_{1,2} \land (P_{2,2} \lor P_{3,1})$?

- $\neg P_{1,2} = \neg false = true$
- $P_{2,2} \lor P_{3,1} = false \lor true = true$
- Gabungkan: $true \land true = true$

Jadi $\neg P_{1,2} \land (P_{2,2} \lor P_{3,1})$ bernilai **true** di bawah
$m_1$. Langkah berikutnya cuma mengecek hitungan manual ini pakai `pl_true`.
Kalau hasilnya tidak sama, berarti ada yang salah di salah satu perhitungan.

In [10]:
psource(pl_true)

In [11]:
# m1 from slide 29. Keys are Expr symbols, not strings.
P12, P22, P31 = expr('P12, P22, P31')
m1 = {P12: False, P22: False, P31: True}

sentence = ~P12 & (P22 | P31)
print(sentence)
print("pl_true:", pl_true(sentence, m1))

(~P12 & (P22 | P31))
pl_true: True


In [12]:
# pl_true needs a value for every symbol in the sentence (the "setiap" from
# Penjelasan). Leave one out and it returns None ("not obvious") instead of
# guessing: an incomplete model can't always determine a value.
partial_model = {P12: False, P22: False}  # P31 missing on purpose
print(pl_true(sentence, partial_model))

None


---
# 3.4 Truth Tables dan Implikasi Material

![Figure 7.8](img/fig-7-8-truth-tables.png)

## Penjelasan

**Truth table** merekap nilai sebuah sentence untuk setiap kombinasi nilai
proposition symbol-nya. Untuk dua simbol $P$ dan $Q$, ada $2^2 = 4$ kombinasi,
satu baris per kombinasi:

| $P$ | $Q$ | $\neg P$ | $P \land Q$ | $P \lor Q$ | $P \Rightarrow Q$ | $P \Leftrightarrow Q$ |
|---|---|---|---|---|---|---|
| false | false | true | false | false | true | true |
| false | true | true | false | true | true | false |
| true | false | false | false | true | false | false |
| true | true | false | true | true | true | true |


Baris **implikasi** ($P \Rightarrow Q$) yang paling perlu diperhatikan: hanya
bernilai salah pada satu kasus, yaitu ketika $P$ benar dan $Q$ salah. Di tiga
kasus lainnya, termasuk ketika $P$ salah, implikasi otomatis bernilai benar.
Tidak ada syarat bahwa $P$ dan $Q$ harus berhubungan makna sama sekali:
inilah yang disebut **implikasi material**.

Aturan ini yang bikin dua pertanyaan berikut sama-sama bernilai benar, walau
alasannya beda:

1. "5 is odd implies Tokyo is the capital of Japan": benar, karena premis
   ($5$ ganjil) dan konklusi (Tokyo ibu kota Jepang) sama-sama benar.
2. "5 is even implies Sam is smart": benar juga, tapi karena premisnya
   salah ($5$ bukan genap). Implikasi dengan premis salah otomatis benar, apa
   pun nilai konklusinya, termasuk kalau Sam sebenarnya tidak pintar.

Kasus kedua ini yang biasa bikin protes, dan wajar. Bantu dengan analogi
janji: "kalau besok hujan, saya bawa payung" cuma dilanggar kalau besok hujan
dan saya tidak bawa payung. Kalau ternyata besok tidak hujan, janji itu tidak
pernah dilanggar, apa pun yang saya lakukan soal payung, persis pola baris
implikasi di atas ketika $P$ salah.

**Vocabulary tambahan.** Truth table sebuah sentence juga bisa dipakai
mengklasifikasikannya:

- **Tautology** (valid): benar di semua baris. Contoh: $P \lor \neg P$.
- **Satisfiable** (consistent): benar di setidaknya satu baris.
- **Unsatisfiable** (inconsistent): tidak pernah benar di baris mana pun.

Ketiganya dicek dengan cara yang sama seperti membaca truth table di atas,
cuma lihat semua baris sekaligus, bukan satu baris tertentu.

## Contoh penerapan

Bangkitkan ulang Figure 7.8 lewat `pl_true`, bukan disalin dari gambar di
atas: loop semua kombinasi $P$ dan $Q$ pakai `itertools.product`, lalu
tampilkan sebagai DataFrame dengan kolom untuk kelima konektif. Hasilnya
harus sama persis dengan gambar.

Dua pertanyaan jebakan tadi juga dijawab pakai kode. Untuk yang kedua,
tunjukkan hasilnya tetap true baik ketika Sam pintar maupun tidak.

In [13]:
# Regenerate Figure 7.8: loop every P, Q combination and evaluate all five
# connectives with pl_true. [False, True] keeps the row order the same as
# the slide (binary count: false before true).
P, Q = expr('P, Q')
connectives = {
    "¬P": ~P,
    "P∧Q": P & Q,
    "P∨Q": P | Q,
    "P⇒Q": expr('P ==> Q'),
    "P⇔Q": expr('P <=> Q'),
}

rows = []
for p_val, q_val in itertools.product([False, True], repeat=2):
    model = {P: p_val, Q: q_val}
    row = {"P": p_val, "Q": q_val}
    row.update({name: pl_true(formula, model) for name, formula in connectives.items()})
    rows.append(row)

table = pd.DataFrame(rows)
table

,P,Q,¬P,P∧Q,P∨Q,P⇒Q,P⇔Q
0,False,False,True,False,False,True,True
1,False,True,True,False,True,True,False
2,True,False,False,False,True,False,False
3,True,True,False,True,True,True,True


In [14]:
# Trick question 1: "5 is odd implies Tokyo is the capital of Japan".
# Both premise and conclusion are established facts, not variables.
five_is_odd = True
tokyo_is_capital_of_japan = True

result_1 = pl_true(expr('P ==> Q'), {P: five_is_odd, Q: tokyo_is_capital_of_japan})
print("5 is odd implies Tokyo is the capital of Japan:", result_1)

5 is odd implies Tokyo is the capital of Japan: True


In [15]:
# Trick question 2: "5 is even implies Sam is smart". Premise is false, so
# the implication should stay true no matter Sam's actual smartness.
five_is_even = False
implication = expr('P ==> Q')

for sam_is_smart in [True, False]:
    result_2 = pl_true(implication, {P: five_is_even, Q: sam_is_smart})
    print(f"Sam is smart = {sam_is_smart!s:5} -> 5 is even implies Sam is smart = {result_2}")

Sam is smart = True  -> 5 is even implies Sam is smart = True
Sam is smart = False -> 5 is even implies Sam is smart = True


In [16]:
# Tie both trick questions back to the big table above. Question 1 lands
# on the P=true row (premise and conclusion both true). Question 2 lands
# on the P=false rows: P=>Q is true there no matter what Q is.
print(table.loc[table["P"], ["P", "Q", "P⇒Q"]])
print(table.loc[~table["P"], ["P", "Q", "P⇒Q"]])

      P      Q    P⇒Q
2  True  False  False
3  True   True   True
       P      Q   P⇒Q
0  False  False  True
1  False   True  True


---
# Latihan Soal

## Soal 1: Diskusi Kelompok

**Slide 27**

> Diketahui proposisi atomic $x_1$, $x_2$, $x_3$ di mana $x_i$ merupakan notasi
> untuk kalimat "pegawai i sedang bekerja". Buatlah proposisi sesuai kondisi
> berikut:
>
> 1. Salah satu dari pegawai 1, 2 dan 3 sedang bekerja.
> 2. Terdapat dua pegawai dari pegawai 1, 2, dan 3 yang sedang bekerja.
> 3. Tidak semua pegawai sedang bekerja.
>
> Jika sudah, diskusikan jawaban kelompokmu dengan kelompok lain.

Terjemahkan tiap kondisi jadi proposisi, lalu cek jawabanmu dengan fungsi
`satisfying_models` di bawah, yang mengembalikan semua model (dari 8 model
yang mungkin) yang membuat proposisimu bernilai benar. Simbolnya ditulis
`X1`, `X2`, `X3` (uppercase) di kode, mewakili $x_1$, $x_2$, $x_3$ di soal.
`pl_true` cuma mengenali simbol yang diawali huruf besar sebagai proposition
symbol.

In [17]:
def satisfying_models(sentence, syms):
    """Enumerate every model over `syms` and return the ones where `sentence` is true."""
    models = []
    for values in itertools.product([False, True], repeat=len(syms)):
        model = dict(zip(syms, values))
        if pl_true(sentence, model):
            models.append(model)
    return models


X1, X2, X3 = expr('X1, X2, X3')
print("Total kemungkinan model:", 2 ** 3)

Total kemungkinan model: 8


<details>
<summary>Klik untuk melihat hint</summary>

Nomor 1 dan 2 sengaja ambigu -- itu bagian dari pelajarannya, jangan
diselesaikan sepihak. Coba tulis **dua** versi proposisi untuk masing-masing
("paling sedikit satu/dua" lawan "tepat satu/dua"), lalu bandingkan
`len(satisfying_models(...))` dari kedua versi itu -- beda tidak?

Nomor 3, hati-hati: "tidak semua bekerja" tidak sama dengan "tidak ada yang
bekerja". Tulis dulu $\neg(x_1 \land x_2 \land x_3)$ apa adanya (jangan buru-
buru disederhanakan), lalu bandingkan jumlah model yang memenuhinya dengan
jumlah model untuk $\neg x_1 \land \neg x_2 \land \neg x_3$.

</details>

## Soal 2

Buktikan hukum De Morgan $\neg(P \land Q) \equiv \neg P \lor \neg Q$ dengan
truth table yang dibangkitkan pakai `pl_true` untuk semua kombinasi $P$ dan
$Q$, bukan dihitung manual.

<details>
<summary>Klik untuk melihat hint</summary>

Bangun `lhs` dan `rhs` sebagai `Expr` sesuai kedua sisi persamaannya, lalu
loop `itertools.product` untuk $P$, $Q$ persis pola yang sama dengan tabel
di 3.4 -- bandingkan `pl_true` keduanya di tiap baris, jangan dihitung
manual dulu. Kalau di semua baris nilainya sama, apa artinya itu buat
kalimat $\text{lhs} \Leftrightarrow \text{rhs}$ (lihat "Vocabulary
tambahan" di 3.4)?

</details>

## Soal 3

Di 3.2 disebutkan implikasi ($\Rightarrow$) adalah konektif yang jadi bentuk
*rule* KB nanti di Notebook 04, misalnya kalimat seperti "kalau ada breeze di
suatu kotak, maka ada pit di salah satu kotak tetangganya." Implikasi
material tidak mensyaratkan premise dan conclusion saling berhubungan makna
(3.4): implikasi otomatis bernilai benar begitu premise-nya salah, berapa
pun nilai conclusion-nya.

Jelaskan dengan kata-katamu sendiri: kenapa sifat ini justru yang membuat
implikasi material cocok dipakai sebagai bentuk rule KB Wumpus World, bukan
gangguan yang perlu dihindari? Bayangkan agent memakai rule "kalau breeze di
$[x,y]$, maka pit di salah satu tetangganya" — apa yang salah dengan KB
seorang agent kalau rule itu justru dianggap **salah** (bukan otomatis
benar) setiap kali agent tidak merasakan breeze di $[x,y]$?

Jawaban tidak cukup satu baris: kaitkan dengan tabel implikasi di 3.4, dan
pakai alasan yang spesifik ke skenario Wumpus, bukan cuma definisi umum
implikasi material.

<details>
<summary>Klik untuk melihat hint</summary>

Bayangkan implikasi didefinisikan ulang supaya bernilai **false** ketika
premise-nya false (kebalikan dari sekarang). KB adalah conjunction dari
semua rule yang di-tell ke dalamnya (Notebook 02). Kalau ada **satu saja**
kotak tanpa breeze, apa yang terjadi pada rule breeze-pit untuk kotak itu
di bawah definisi baru ini -- dan lewat operator AND, apa yang terjadi ke
**seluruh** KB?

Cek juga definisi entailment dari Notebook 02, $M(KB) \subseteq
M(\alpha)$: apa yang terjadi pada definisi itu kalau $M(KB)$ ternyata
kosong di setiap model?

</details>